# F1 pole position predictor

## Import


In [92]:
import pandas as pd
import numpy as np
import sklearn
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 10)

qualifying = pd.read_csv("F1(1950 - 2024) data\\raw\\qualifying.csv" , na_values=['\\N'])
races = pd.read_csv("F1(1950 - 2024) data\\raw\\races.csv", na_values=['\\N'])
drivers = pd.read_csv("F1(1950 - 2024) data\\raw\\drivers.csv", na_values=['\\N'])
constructors = pd.read_csv("F1(1950 - 2024) data\\raw\\constructors.csv", na_values=['\\N'])

print('Data loaded successfully')

Data loaded successfully


## Checking data

In [93]:
print(qualifying.shape)
qualifying.head()

(10494, 9)


,qualifyId,raceId,driverId,constructorId,number,position,q1,q2,q3
0,1,18,1,1,22,1,1:26.572,1:25.187,1:26.714
1,2,18,9,2,4,2,1:26.103,1:25.315,1:26.869
2,3,18,5,1,23,3,1:25.664,1:25.452,1:27.079
3,4,18,13,6,2,4,1:25.994,1:25.691,1:27.178
4,5,18,2,2,3,5,1:25.960,1:25.518,1:27.236


## check null

In [94]:
qualifying.isnull().sum()

qualifyId           0
raceId              0
driverId            0
constructorId       0
number              0
position            0
q1                156
q2               4647
q3               6865
dtype: int64

In [95]:
races.sample(5)

,raceId,year,round,circuitId,name,date,time,url,fp1_date,fp1_time,fp2_date,fp2_time,fp3_date,fp3_time,quali_date,quali_time,sprint_date,sprint_time
923,933,2015,8,70,Austrian Grand Prix,2015-06-21,12:00:00,http://en.wikipedia.org/wiki/2015_Austrian_Gra...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
828,829,1951,5,9,British Grand Prix,1951-07-14,NaN,http://en.wikipedia.org/wiki/1951_British_Gran...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
820,821,1952,5,9,British Grand Prix,1952-07-19,NaN,http://en.wikipedia.org/wiki/1952_British_Gran...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
817,818,1952,2,19,Indianapolis 500,1952-05-30,NaN,http://en.wikipedia.org/wiki/1952_Indianapolis...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
833,834,1950,2,6,Monaco Grand Prix,1950-05-21,NaN,http://en.wikipedia.org/wiki/1950_Monaco_Grand...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Data manipulation

In [96]:
df = pd.merge(qualifying, races, how='left', on='raceId')
df.sample(5)

,qualifyId,raceId,driverId,constructorId,number,position,q1,q2,q3,year,round,circuitId,name,date,time,url,fp1_date,fp1_time,fp2_date,fp2_time,fp3_date,fp3_time,quali_date,quali_time,sprint_date,sprint_time
8940,8981,1064,839,214,31,8,1:10.179,1:09.919,1:09.933,2021,13,39,Dutch Grand Prix,2021-09-05,13:00:00,http://en.wikipedia.org/wiki/2021_Dutch_Grand_...,2021-09-03,NaN,2021-09-03,NaN,2021-09-04,NaN,2021-09-04,NaN,NaN,NaN
3679,3681,351,810,166,25,20,1:51.107,NaN,NaN,2010,15,15,Singapore Grand Prix,2010-09-26,12:00:00,http://en.wikipedia.org/wiki/2010_Singapore_Gr...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4052,4054,122,17,19,14,14,1:13.269,NaN,NaN,2003,15,19,United States Grand Prix,2003-09-28,NaN,http://en.wikipedia.org/wiki/2003_United_State...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1061,1062,67,3,3,10,12,1:22.581,1:22.203,NaN,2006,15,14,Italian Grand Prix,2006-09-10,14:00:00,http://en.wikipedia.org/wiki/2006_Italian_Gran...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3229,3231,13,18,23,22,6,1:23.403,1:22.955,1:25.030,2009,13,14,Italian Grand Prix,2009-09-13,12:00:00,http://en.wikipedia.org/wiki/2009_Italian_Gran...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Drop pre 2005

In [97]:
df = df[df['year'] >= 2005]

## Convert Str to seconds

In [98]:
df['q1'].dtypes

<StringDtype(storage='python', na_value=nan)>

In [99]:
df['q1'] = df['q1'].str.split(':').str[0].astype(float) * 60 + df['q1'].str.split(':').str[1].astype(float)

df['q2'] = df['q2'].str.split(':').str[0].astype(float) * 60 + df['q2'].str.split(':').str[1].astype(float)

df['q3'] = df['q3'].str.split(':').str[0].astype(float) * 60 + df['q3'].str.split(':').str[1].astype(float)


In [100]:
df['q1'].head(10)

0    86.572
1    86.103
2    85.664
3    85.994
4    85.960
5    86.427
6    86.295
7    86.381
8    86.919
9    86.702
Name: q1, dtype: float64

## Filling nulls with per year medians

In [101]:
df['q1'] = df.groupby('year')['q1'].transform(lambda x: x.fillna(x.median()))
df['q2'] = df.groupby('year')['q2'].transform(lambda x: x.fillna(x.median()))
df['q3'] = df.groupby('year')['q3'].transform(lambda x: x.fillna(x.median()))


In [102]:
df[['q1', 'q2', 'q3']].isnull().sum()

q1      0
q2      0
q3    376
dtype: int64

## Filling q3 nulls with 0

In [103]:
df['q3'] = df['q3'].fillna(0)

## create reached_q2&q3 column

In [104]:
df['reached_q2'] = (df['q2'].notnull()).astype(int)
df['reached_q3'] = (df['q3'].notnull()).astype(int)
df.sample(5)

,qualifyId,raceId,driverId,constructorId,number,position,q1,q2,q3,year,round,circuitId,name,date,time,url,fp1_date,fp1_time,fp2_date,fp2_time,fp3_date,fp3_time,quali_date,quali_time,sprint_date,sprint_time,reached_q2,reached_q3
5720,5723,893,813,3,16,18,106.619,91.989,90.9080,2013,13,15,Singapore Grand Prix,2013-09-22,12:00:00,http://en.wikipedia.org/wiki/2013_Singapore_Gr...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,1
6658,6681,945,829,209,28,19,106.297,93.361,89.4800,2015,19,24,Abu Dhabi Grand Prix,2015-11-29,13:00:00,http://en.wikipedia.org/wiki/2015_Abu_Dhabi_Gr...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,1
3369,3371,338,810,166,25,22,90.185,91.155,90.7050,2010,2,1,Australian Grand Prix,2010-03-28,06:00:00,http://en.wikipedia.org/wiki/2010_Australian_G...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,1
9927,9985,1116,822,51,77,13,96.082,95.858,84.9105,2023,18,69,United States Grand Prix,2023-10-22,19:00:00,https://en.wikipedia.org/wiki/2023_United_Stat...,2023-10-20,17:30:00,2023-10-21,18:00:00,NaN,NaN,2023-10-20,21:00:00,2023-10-21,22:00:00,1,1
10157,10215,1128,846,1,4,4,71.760,70.732,70.5420,2024,8,6,Monaco Grand Prix,2024-05-26,13:00:00,https://en.wikipedia.org/wiki/2024_Monaco_Gran...,2024-05-24,11:30:00,2024-05-24,15:00:00,2024-05-25,10:30:00,2024-05-25,14:00:00,NaN,NaN,1,1


In [105]:
print(df[df['q3'].isnull()]['reached_q3'].value_counts())

Series([], Name: count, dtype: int64)


## create got_pole column

In [106]:
df['got_pole'] = (df['position'] == 1).astype(int)
df.sample(5)

,qualifyId,raceId,driverId,constructorId,number,position,q1,q2,q3,year,round,circuitId,name,date,time,url,fp1_date,fp1_time,fp2_date,fp2_time,fp3_date,fp3_time,quali_date,quali_time,sprint_date,sprint_time,reached_q2,reached_q3,got_pole
4682,4685,847,3,131,8,6,74.920,73.9500,73.8140,2011,7,7,Canadian Grand Prix,2011-06-12,17:00:00,http://en.wikipedia.org/wiki/2011_Canadian_Gra...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,1,0
206,207,27,16,10,20,19,76.657,83.4555,86.7915,2008,10,10,German Grand Prix,2008-07-20,12:00:00,http://en.wikipedia.org/wiki/2008_German_Grand...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,1,0
6343,6366,929,1,131,44,1,93.928,92.6690,92.5710,2015,4,3,Bahrain Grand Prix,2015-04-19,15:00:00,http://en.wikipedia.org/wiki/2015_Bahrain_Gran...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,1,1
4783,4786,851,814,10,15,11,82.976,82.2560,90.3075,2011,11,11,Hungarian Grand Prix,2011-07-31,12:00:00,http://en.wikipedia.org/wiki/2011_Hungarian_Gr...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,1,0
8727,8768,1053,849,3,6,14,75.653,75.5930,81.0545,2021,2,21,Emilia Romagna Grand Prix,2021-04-18,13:00:00,http://en.wikipedia.org/wiki/2021_Emilia_Romag...,2021-04-16,NaN,2021-04-16,NaN,2021-04-17,NaN,2021-04-17,NaN,NaN,NaN,1,1,0


In [107]:
df['got_pole'].value_counts()

got_pole
0    7872
1     394
Name: count, dtype: int64

In [108]:
df.groupby('year')['q2'].median().head(20)

year
2005    89.1590
2006    81.4840
2007    83.1205
2008    83.4555
2009    91.2220
         ...   
2020    79.9620
2021    81.0860
2022    85.2250
2023    84.4405
2024    83.8770
Name: q2, Length: 20, dtype: float64

In [109]:
for i, col in enumerate(df.columns):
    print(i, col)

pd.set_option('display.max_columns', None)
df.sample(20)

0 qualifyId
1 raceId
2 driverId
3 constructorId
4 number
5 position
6 q1
7 q2
8 q3
9 year
10 round
11 circuitId
12 name
13 date
14 time
15 url
16 fp1_date
17 fp1_time
18 fp2_date
19 fp2_time
20 fp3_date
21 fp3_time
22 quali_date
23 quali_time
24 sprint_date
25 sprint_time
26 reached_q2
27 reached_q3
28 got_pole


,qualifyId,raceId,driverId,constructorId,number,position,q1,q2,q3,year,round,circuitId,name,date,time,url,fp1_date,fp1_time,fp2_date,fp2_time,fp3_date,fp3_time,quali_date,quali_time,sprint_date,sprint_time,reached_q2,reached_q3,got_pole
5728,5731,894,154,208,8,4,99.065,98.0760,97.5310,2013,14,35,Korean Grand Prix,2013-10-06,06:00:00,http://en.wikipedia.org/wiki/2013_Korean_Grand...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,1,0
9009,9050,1067,832,6,55,15,85.177,81.0860,81.0545,2021,16,5,Turkish Grand Prix,2021-10-10,12:00:00,http://en.wikipedia.org/wiki/2021_Turkish_Gran...,2021-10-08,NaN,2021-10-08,NaN,2021-10-09,NaN,2021-10-09,NaN,NaN,NaN,1,1,0
267,268,30,21,10,21,20,108.447,83.4555,86.7915,2008,13,13,Belgian Grand Prix,2008-09-07,12:00:00,http://en.wikipedia.org/wiki/2008_Belgian_Gran...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,1,0
1315,1316,85,38,18,20,18,84.904,89.1590,0.0000,2005,15,14,Italian Grand Prix,2005-09-04,14:00:00,http://en.wikipedia.org/wiki/2005_Italian_Gran...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,1,0
5231,5234,871,17,9,2,7,109.859,108.5460,108.3920,2012,12,13,Belgian Grand Prix,2012-09-02,12:00:00,http://en.wikipedia.org/wiki/2012_Belgian_Gran...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3734,3736,354,17,9,6,3,79.025,78.5160,75.6370,2010,18,18,Brazilian Grand Prix,2010-11-07,16:00:00,http://en.wikipedia.org/wiki/2010_Brazilian_Gr...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,1,0
8270,8294,1026,815,211,11,17,90.344,86.1605,85.7870,2019,17,22,Japanese Grand Prix,2019-10-13,05:10:00,http://en.wikipedia.org/wiki/2019_Japanese_Gra...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,1,0
10153,10211,1127,858,3,2,20,87.772,83.8770,83.2340,2024,7,21,Emilia Romagna Grand Prix,2024-05-19,13:00:00,https://en.wikipedia.org/wiki/2024_Emilia_Roma...,2024-05-17,11:30:00,2024-05-17,15:00:00,2024-05-18,10:30:00,2024-05-18,14:00:00,NaN,NaN,1,1,0
8168,8192,1021,825,210,20,15,76.122,77.0810,85.7870,2019,12,11,Hungarian Grand Prix,2019-08-04,13:10:00,http://en.wikipedia.org/wiki/2019_Hungarian_Gr...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,1,0


## Features and Target

In [110]:
features = ['q1', 'q2', 'q3', 'reached_q2', 'reached_q3', 'year', 'circuitId', 'constructorId', 'driverId', 'round']
target = 'got_pole'

## New df

In [111]:
df_cleaned = df[features + [target]]
print(df_cleaned.shape)
df_cleaned.sample(5)

(8266, 11)


,q1,q2,q3,reached_q2,reached_q3,year,circuitId,constructorId,driverId,round,got_pole
3446,75.110,74.5680,74.2270,1,1,2010,6,9,20,6,0
5885,118.883,120.8390,121.7120,1,1,2014,2,10,807,2,0
6056,72.673,95.0920,94.4715,1,1,2014,70,207,828,8,0
585,93.151,83.1205,83.4460,1,1,2007,20,9,14,10,0
1166,76.201,89.1590,0.0000,1,1,2005,7,7,15,8,0


In [112]:
print(df_cleaned.shape)
print(df_cleaned['got_pole'].value_counts())
print(df_cleaned.isnull().sum())

(8266, 11)
got_pole
0    7872
1     394
Name: count, dtype: int64
q1               0
q2               0
q3               0
reached_q2       0
reached_q3       0
                ..
circuitId        0
constructorId    0
driverId         0
round            0
got_pole         0
Length: 11, dtype: int64


## sklearn

In [113]:
from sklearn.model_selection import train_test_split

x = df_cleaned[features]
y = df_cleaned[target]

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

print(x_train.shape, x_test.shape)


(6612, 10) (1654, 10)


## Logistic Regression

In [114]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(class_weight='balanced',max_iter=1000)
model.fit(x_train, y_train)
print(model.classes_)

[0 1]


c:\Users\Tahsin\anaconda3\envs\data_analysis\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [115]:
from sklearn.metrics import classification_report

y_pred = model.predict(x_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.98      0.59      0.74      1575
           1       0.08      0.75      0.15        79

    accuracy                           0.60      1654
   macro avg       0.53      0.67      0.44      1654
weighted avg       0.94      0.60      0.71      1654



## Random Forrest

In [116]:
from sklearn.ensemble import RandomForestClassifier

model_rf = RandomForestClassifier(class_weight='balanced', random_state=42)
model_rf.fit(x_train, y_train)

y_pred_rf = model_rf.predict(x_test)
print(classification_report(y_test, y_pred_rf))

              precision    recall  f1-score   support

           0       0.96      1.00      0.98      1575
           1       0.56      0.06      0.11        79

    accuracy                           0.95      1654
   macro avg       0.76      0.53      0.54      1654
weighted avg       0.94      0.95      0.93      1654



Chose Logistic Regression over Random Forest despite lower accuracy (60% vs 95%).

Random Forest only caught 6% of actual poles - too many silent misses.

Logistic Regression catches 75% of poles; false alarms are low-cost since a human reviews flagged candidates, but a missed pole means it's never investigated at all.

In [117]:
y_proba = model.predict_proba(x_test)[:, 1]

threshold = 0.4
y_pred_custom = (y_proba >= threshold).astype(int)

print(classification_report(y_test, y_pred_custom))

              precision    recall  f1-score   support

           0       0.98      0.40      0.57      1575
           1       0.07      0.84      0.12        79

    accuracy                           0.42      1654
   macro avg       0.52      0.62      0.35      1654
weighted avg       0.94      0.42      0.55      1654



In [118]:
thresholds= np.arange(0.1, 0.9, 0.05)

for t in thresholds:
    y_pred_t = (y_proba >= t).astype(int)
    precision = (y_pred_t & y_test.values).sum() / y_pred_t.sum() if y_pred_t.sum() > 0 else 0
    recall = (y_pred_t & y_test.values).sum() / y_test.sum()
    print(f'threshold: {t:.2f}, Precision: {precision:.2f}, Recall: {recall:.2f}')

threshold: 0.10, Precision: 0.05, Recall: 1.00
threshold: 0.15, Precision: 0.05, Recall: 0.99
threshold: 0.20, Precision: 0.05, Recall: 0.97
threshold: 0.25, Precision: 0.05, Recall: 0.96
threshold: 0.30, Precision: 0.06, Recall: 0.96
threshold: 0.35, Precision: 0.06, Recall: 0.92
threshold: 0.40, Precision: 0.07, Recall: 0.84
threshold: 0.45, Precision: 0.07, Recall: 0.82
threshold: 0.50, Precision: 0.08, Recall: 0.75
threshold: 0.55, Precision: 0.09, Recall: 0.56
threshold: 0.60, Precision: 0.09, Recall: 0.37
threshold: 0.65, Precision: 0.07, Recall: 0.19
threshold: 0.70, Precision: 0.06, Recall: 0.11
threshold: 0.75, Precision: 0.04, Recall: 0.05
threshold: 0.80, Precision: 0.02, Recall: 0.01
threshold: 0.85, Precision: 0.00, Recall: 0.00


In [119]:
print(df_cleaned['driverId'].nunique())
print(df_cleaned['constructorId'].nunique())
print(df_cleaned['circuitId'].nunique())

102
34
38


## Rebuild features

In [120]:
features_v2 = ['q1', 'q2', 'q3', 'reached_q2', 'reached_q3', 'year', 'round', 'circuitId']

df_v2 = df_cleaned[features_v2 + ['got_pole']].copy()

df_v2 = pd.get_dummies(df_v2, columns=['circuitId'], prefix='circuit')

print(df_v2.shape)
df_v2.head()

(8266, 46)


,q1,q2,q3,reached_q2,reached_q3,year,round,got_pole,circuit_1,circuit_2,circuit_3,circuit_4,circuit_5,circuit_6,circuit_7,circuit_8,circuit_9,circuit_10,circuit_11,circuit_12,circuit_13,circuit_14,circuit_15,circuit_16,circuit_17,circuit_18,circuit_19,circuit_20,circuit_21,circuit_22,circuit_24,circuit_32,circuit_34,circuit_35,circuit_39,circuit_68,circuit_69,circuit_70,circuit_71,circuit_73,circuit_75,circuit_76,circuit_77,circuit_78,circuit_79,circuit_80
0,86.572,85.187,86.714,1,1,2008,1,1,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
1,86.103,85.315,86.869,1,1,2008,1,0,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
2,85.664,85.452,87.079,1,1,2008,1,0,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
3,85.994,85.691,87.178,1,1,2008,1,0,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
4,85.960,85.518,87.236,1,1,2008,1,0,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False


In [121]:
features_v2 = df_v2.drop(columns=['got_pole']).columns.tolist()
target_v2 = 'got_pole'

In [122]:
f = df_v2[features_v2]
t = df_v2[target_v2]

f_train, f_test, t_train, t_test = train_test_split(f, t, test_size=0.2, random_state=42)

print(f_train.shape, f_test.shape)

(6612, 45) (1654, 45)


In [124]:
model_v2 = LogisticRegression(class_weight='balanced', max_iter=1000)
model_v2.fit(f_train, t_train)

t_pred = model_v2.predict(f_test)
print(classification_report(t_test, t_pred))

              precision    recall  f1-score   support

           0       0.94      0.50      0.65      1575
           1       0.04      0.41      0.07        79

    accuracy                           0.49      1654
   macro avg       0.49      0.45      0.36      1654
weighted avg       0.90      0.49      0.62      1654



c:\Users\Tahsin\anaconda3\envs\data_analysis\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
